# Design a DynamoDB table with a GSI and prove a sparse-index query is cheaper than a scan

A SaaS subscription product has 10,000 tenants and 250,000 subscription records, and the active-subscription lookup runs every time a tenant signs in. The previous engineer wrote a `Scan` with a `FilterExpression: tenant_id = :tid` and a `LastEvaluatedKey` loop. The DynamoDB bill climbed past $400/month with provisioned capacity throttled red, and the cofounder wants this fixed before the next pricing cycle. You have one session to design the right GSI, prove the query path consumes orders of magnitude less capacity than the scan path, and hand the team the access-pattern documentation.

**Time.** Plan for about 45 minutes hands-on. DynamoDB tables come up in seconds; the bulk of the lab is in writing the right keys and reading the ConsumedCapacity numbers.

**Cost.** DynamoDB on-demand pricing means you pay only for what you actually call. 100 writes, two reads, and one scan across 100 items is fractions of a penny in total. The expensive version of this workload is the one running in production right now with a Scan on every signin; the whole point of this lab is to make that bill stop growing.

This lab maps to AWS SAA-C03 Domain 3: Design High-Performing Architectures (24% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.5.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json
import time
from decimal import Decimal

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "dynamodb-gsi-and-access-patterns"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

TABLE_NAME = f"labrun-{LAB_ID}"
GSI_NAME = "tenant-index"
TARGET_TENANT = "T1"   # 5 items belong to this tenant
OTHER_TENANTS = [f"T{i}" for i in range(2, 21)]  # 19 other tenants, 5 items each = 95
TOTAL_ITEMS = 100

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, confirm region.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA Batch 1 labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"Table: {TABLE_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan. Single resource: the
# DynamoDB table. The GSI and items are destroyed when the table is deleted.

CLEANUP_MANIFEST: list[CleanupResource] = [
    CleanupResource(
        type="dynamodb_table",
        id=TABLE_NAME,
        region=REGION,
        cli_delete_command=f"aws dynamodb delete-table --table-name {TABLE_NAME}",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the table with composite primary key, on-demand billing, and a GSI on tenant_id

DynamoDB tables are immutable in their key schema: you cannot add a partition key or sort key after `create_table` returns. You also cannot change BillingMode without disruption. Get both right at create time.

Build it in this order:

1. Create the table in one call with a composite primary key (`pk` HASH, `sk` RANGE), `BillingMode=PAY_PER_REQUEST` (on-demand), and a Global Secondary Index named `tenant-index` keyed on `tenant_id` (HASH) and `created_at` (RANGE) with `ProjectionType=ALL`. Tag the table with the lab tag.
2. Wait for `TableStatus=ACTIVE` using the table-exists waiter.
3. Wait for `GlobalSecondaryIndexes[0].IndexStatus=ACTIVE` (the GSI status is a separate field that the waiter does NOT cover).

Creating the GSI inline with `create_table` is cheaper than adding it later via `update_table` because the table is empty; backfilling a GSI on an existing 250k-item table is the expensive failure mode the cofounder is trying to avoid.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the table with composite primary key + on-demand + GSI inline.

ddb = boto3.client(
    "dynamodb",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Call ddb.create_table() with these arguments:
#   TableName=TABLE_NAME,
#   AttributeDefinitions=[
#       {"AttributeName": "pk", "AttributeType": "S"},
#       {"AttributeName": "sk", "AttributeType": "S"},
#       {"AttributeName": "tenant_id", "AttributeType": "S"},
#       {"AttributeName": "created_at", "AttributeType": "S"},
#   ],
#   KeySchema=[
#       {"AttributeName": "pk", "KeyType": "HASH"},
#       {"AttributeName": "sk", "KeyType": "RANGE"},
#   ],
#   BillingMode="PAY_PER_REQUEST",
#   GlobalSecondaryIndexes=[
#       {
#           "IndexName": GSI_NAME,
#           "KeySchema": [
#               {"AttributeName": "tenant_id", "KeyType": "HASH"},
#               {"AttributeName": "created_at", "KeyType": "RANGE"},
#           ],
#           "Projection": {"ProjectionType": "ALL"},
#       }
#   ],
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],

print(f"create_table returned. Waiting for ACTIVE...")

waiter = ddb.get_waiter("table_exists")
waiter.wait(TableName=TABLE_NAME, WaiterConfig={"Delay": 5, "MaxAttempts": 36})
print(f"Table {TABLE_NAME} is ACTIVE.")

# Poll for GSI ACTIVE. The table-exists waiter does not cover GSI status.
print("Polling for GSI ACTIVE (typically 30 to 90 seconds on an empty table)...")
wait_messages = [
    "  still building, hold tight...",
    "  GSI backfill is fast on an empty table, almost there...",
    "  any second now...",
]
msg_i = 0
elapsed = 0
while elapsed < 300:
    desc = ddb.describe_table(TableName=TABLE_NAME)
    gsis = desc.get("Table", {}).get("GlobalSecondaryIndexes", [])
    if gsis and gsis[0].get("IndexStatus") == "ACTIVE":
        print(f"GSI {GSI_NAME} is ACTIVE.")
        break
    print(wait_messages[msg_i % len(wait_messages)])
    msg_i += 1
    time.sleep(5)
    elapsed += 5
else:
    print("GSI did not reach ACTIVE within 5 minutes. Re-run this cell.")
    raise SystemExit(1)

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Table exists with composite primary key (pk HASH + sk RANGE),
# BillingMode=PAY_PER_REQUEST, TableStatus=ACTIVE, and carries the lab tag.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        ddb_client = boto3.client(
            "dynamodb",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            desc = ddb_client.describe_table(TableName=TABLE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {TABLE_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        table = desc["Table"]
        status = table.get("TableStatus")
        if status != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=f"Table {TABLE_NAME} status is {status!r}, expected 'ACTIVE'.",
            )
        ks = {k["AttributeName"]: k["KeyType"] for k in table.get("KeySchema", [])}
        if ks.get("pk") != "HASH" or ks.get("sk") != "RANGE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table key schema is {ks!r}, expected pk=HASH and sk=RANGE. "
                    f"DynamoDB does not allow changing the key schema after create_table."
                ),
            )
        billing = table.get("BillingModeSummary", {}).get("BillingMode")
        if billing != "PAY_PER_REQUEST":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"BillingMode is {billing!r}, expected 'PAY_PER_REQUEST' (on-demand). "
                    f"This lab uses on-demand because the 100-request workload makes provisioned over-spec."
                ),
            )
        try:
            tag_resp = ddb_client.list_tags_of_resource(ResourceArn=table["TableArn"])
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        tags = {t["Key"]: t["Value"] for t in tag_resp.get("Tags", [])}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {tags}"
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint failure message tells you which knob is off: missing table, wrong key schema, wrong billing mode, missing tag. DynamoDB will not let you fix the key schema after create_table; you have to delete and recreate.

</details>

<details><summary>Hint 2 (direction)</summary>

Three parts of `create_table` matter most: `AttributeDefinitions` (list every key attribute used in BOTH the base table key and the GSI key), `KeySchema` with pk as HASH and sk as RANGE, and `BillingMode="PAY_PER_REQUEST"`. Forgetting to include `tenant_id` and `created_at` in AttributeDefinitions is the most common create_table failure because the GSI cannot key on attributes not declared.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the YOUR CODE block):

```python
ddb.create_table(
    TableName=TABLE_NAME,
    AttributeDefinitions=[
        {"AttributeName": "pk", "AttributeType": "S"},
        {"AttributeName": "sk", "AttributeType": "S"},
        {"AttributeName": "tenant_id", "AttributeType": "S"},
        {"AttributeName": "created_at", "AttributeType": "S"},
    ],
    KeySchema=[
        {"AttributeName": "pk", "KeyType": "HASH"},
        {"AttributeName": "sk", "KeyType": "RANGE"},
    ],
    BillingMode="PAY_PER_REQUEST",
    GlobalSecondaryIndexes=[
        {
            "IndexName": GSI_NAME,
            "KeySchema": [
                {"AttributeName": "tenant_id", "KeyType": "HASH"},
                {"AttributeName": "created_at", "KeyType": "RANGE"},
            ],
            "Projection": {"ProjectionType": "ALL"},
        }
    ],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Still under one cent. The empty table at rest costs $0/hour on on-demand. The GSI backfill on an empty table is free.

## Task 2: Verify the GSI shape

The GSI was declared inline with `create_table` in Task 1. This task verifies that the index shape AWS actually built matches what the lab needs. The checkpoint walks `describe_table` and confirms the index name, key schema, projection type, and status are right.

Nothing to write here; the next code cell prints the GSI metadata for your inspection. If the checkpoint fails, you need to drop and recreate the table from Task 1 with the right GSI definition.

In [ ]:
# NBVAL_SKIP
# Task 2: Inspect the GSI metadata. No YOUR CODE here; the cell prints the
# GSI shape so the student can see what describe_table returns.

ddb = boto3.client(
    "dynamodb",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

desc = ddb.describe_table(TableName=TABLE_NAME)
gsis = desc.get("Table", {}).get("GlobalSecondaryIndexes", [])
if not gsis:
    print("No GSI found. Did Task 1 declare GlobalSecondaryIndexes correctly?")
else:
    for gsi in gsis:
        print(f"IndexName:     {gsi.get('IndexName')}")
        print(f"KeySchema:     {gsi.get('KeySchema')}")
        print(f"Projection:    {gsi.get('Projection')}")
        print(f"IndexStatus:   {gsi.get('IndexStatus')}")
        print(f"IndexArn:      {gsi.get('IndexArn')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: GSI tenant-index exists keyed on tenant_id (HASH) and
# created_at (RANGE) with ProjectionType=ALL and IndexStatus=ACTIVE.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        ddb_client = boto3.client(
            "dynamodb",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        desc = ddb_client.describe_table(TableName=TABLE_NAME)
        gsis = desc.get("Table", {}).get("GlobalSecondaryIndexes", [])
        target = next((g for g in gsis if g.get("IndexName") == GSI_NAME), None)
        if target is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GSI {GSI_NAME!r} does not exist on table {TABLE_NAME}. "
                    f"Recreate the table with the GSI declared inline in create_table."
                ),
            )
        ks = {k["AttributeName"]: k["KeyType"] for k in target.get("KeySchema", [])}
        if ks.get("tenant_id") != "HASH" or ks.get("created_at") != "RANGE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GSI key schema is {ks!r}, expected tenant_id=HASH and created_at=RANGE."
                ),
            )
        projection = target.get("Projection", {}).get("ProjectionType")
        if projection != "ALL":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GSI Projection.ProjectionType is {projection!r}, expected 'ALL' so "
                    f"Queries do not need follow-up GetItems on the base table."
                ),
            )
        if target.get("IndexStatus") != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GSI IndexStatus is {target.get('IndexStatus')!r}, expected 'ACTIVE'. "
                    f"Wait for the backfill to finish and re-run this checkpoint."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the checkpoint says the GSI does not exist or has the wrong shape, you need to drop and recreate the table; DynamoDB only allows adding a GSI to a live table via `update_table` (slow + expensive on a populated table), and changing an existing GSI's key schema is not supported.

</details>

<details><summary>Hint 2 (direction)</summary>

The GSI key schema in Task 1 must list `tenant_id` first with `KeyType=HASH` and `created_at` second with `KeyType=RANGE`. ProjectionType must be `ALL` for this lab (one of `ALL`, `KEYS_ONLY`, or `INCLUDE`). If you got the shape wrong, run `ddb.delete_table(TableName=TABLE_NAME)`, wait for it to disappear, and re-run Task 1.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If Checkpoint 1 passed and Checkpoint 2 failed because the GSI definition is wrong, drop and recreate:

```python
ddb.delete_table(TableName=TABLE_NAME)
waiter = ddb.get_waiter("table_not_exists")
waiter.wait(TableName=TABLE_NAME)
# Then re-run the Task 1 cell with the correct GSI definition.
```

</details>

**Wallet check.** Still under one cent. `describe_table` is a free control-plane call.

## Task 3: Load 100 sample subscription items via batch_write_item

The lab needs 100 items spread across 20 tenants: 5 items for the target tenant `T1`, and 5 items each for 19 other tenants. This creates a sparse-index lookup pattern: a GSI Query on `tenant_id = "T1"` reads 5 items along a key range, while a Scan with `FilterExpression: tenant_id = :tid` reads all 100 items before applying the filter.

`batch_write_item` accepts up to 25 PutRequests per call. 100 items = 4 batches of 25.

Build it in this order:

1. Generate 100 items. Each item has `pk` (subscription ID), `sk` (`SUB#<id>`), `tenant_id`, `created_at`, and `plan`.
2. Group the items into 4 batches of 25.
3. Call `batch_write_item` once per batch and handle any `UnprocessedItems` returned.
4. Verify the table now has 100 items via a `scan(Select="COUNT")` call.

In [ ]:
# NBVAL_SKIP
# Task 3: Generate 100 items and load them via batch_write_item in 4 batches of 25.

ddb = boto3.client(
    "dynamodb",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

def generate_items() -> list[dict]:
    """5 items for T1 + 5 items each for T2..T20 = 100 items total."""
    items: list[dict] = []
    counter = 0
    for tenant in [TARGET_TENANT] + OTHER_TENANTS:
        for n in range(5):
            counter += 1
            items.append({
                "pk": {"S": f"SUB#{counter:04d}"},
                "sk": {"S": f"SUB#{counter:04d}"},
                "tenant_id": {"S": tenant},
                "created_at": {"S": f"2026-05-{n+1:02d}T12:00:00Z"},
                "plan": {"S": "pro" if n % 2 == 0 else "starter"},
            })
    return items


items = generate_items()
assert len(items) == TOTAL_ITEMS, f"expected {TOTAL_ITEMS} items, got {len(items)}"
print(f"Generated {len(items)} items across {len(OTHER_TENANTS) + 1} tenants.")

# Split into batches of 25.
batches = [items[i:i + 25] for i in range(0, len(items), 25)]
assert len(batches) == 4

# YOUR CODE: For each batch, call ddb.batch_write_item() with
# RequestItems={TABLE_NAME: [{"PutRequest": {"Item": it}} for it in batch]}.
# Handle the response's UnprocessedItems by retrying until empty (the lab
# payload is small enough that one retry is usually sufficient).

# Verify count.
count_resp = ddb.scan(TableName=TABLE_NAME, Select="COUNT")
print(f"Table item count after load: {count_resp['Count']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Table contains 100 items, and 5 of them belong to TARGET_TENANT.
# Wraps the verification Query in a retry loop because the GSI is eventually
# consistent right after batch_write_item.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        ddb_client = boto3.client(
            "dynamodb",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        count_resp = ddb_client.scan(TableName=TABLE_NAME, Select="COUNT")
        total = count_resp.get("Count", 0)
        if total != TOTAL_ITEMS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table has {total} items, expected {TOTAL_ITEMS}. "
                    f"Did all 4 batches succeed? Check the batch_write_item response "
                    f"for UnprocessedItems and retry any that came back."
                ),
            )

        # GSI eventual consistency retry loop.
        target_count = 0
        elapsed = 0
        while elapsed < 30:
            q = ddb_client.query(
                TableName=TABLE_NAME,
                IndexName=GSI_NAME,
                KeyConditionExpression="tenant_id = :tid",
                ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
                Select="COUNT",
            )
            target_count = q.get("Count", 0)
            if target_count == 5:
                break
            time.sleep(3)
            elapsed += 3

        if target_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GSI Query for tenant_id={TARGET_TENANT} returned {target_count} items, "
                    f"expected 5. Verify the generated payload assigned exactly 5 items to "
                    f"the target tenant."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes: the total count is wrong (some batches did not finish) or the target-tenant count is wrong (the generator did not assign 5 items to T1). The checkpoint message tells you which.

</details>

<details><summary>Hint 2 (direction)</summary>

`batch_write_item` returns `UnprocessedItems` for any PutRequests it could not complete (rare on a 25-item batch but possible under throttling). The pattern is to call in a loop, feed `UnprocessedItems` back as the next `RequestItems`, and exit when the response has none. The GSI is eventually consistent, so the count may briefly read low; the validator already retries for 30 seconds.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 3:

```python
for batch in batches:
    request_items = {TABLE_NAME: [{"PutRequest": {"Item": it}} for it in batch]}
    while request_items:
        resp = ddb.batch_write_item(RequestItems=request_items)
        request_items = resp.get("UnprocessedItems") or None
```

</details>

**Wallet check.** 100 writes at on-demand pricing is $1.25 per million = $0.000125 total. Still well under one cent.

## Task 4: Prove the GSI Query consumes strictly less capacity than the Scan

Both calls return the same 5 items. The point is to surface the underlying capacity cost so the team can see WHY one design beats the other at scale.

Build it in this order:

1. Run a `Query` against the GSI with `KeyConditionExpression: tenant_id = :tid`, `:tid` set to `TARGET_TENANT`, and `ReturnConsumedCapacity="TOTAL"`. Capture `ConsumedCapacity.CapacityUnits` as `q_capacity`.
2. Run a `Scan` against the base table with `FilterExpression: tenant_id = :tid`, `:tid` set to `TARGET_TENANT`, and `ReturnConsumedCapacity="TOTAL"`. Capture `ConsumedCapacity.CapacityUnits` as `s_capacity`.
3. Print both numbers so the student can see the ratio. The Query typically lands around 0.5 RCU; the Scan lands around 12 RCU. The order-of-magnitude gap is the lesson.

In [ ]:
# NBVAL_SKIP
# Task 4: Run the same logical lookup as a GSI Query and as a Scan + Filter.
# Capture ConsumedCapacity from both and print the ratio.

ddb = boto3.client(
    "dynamodb",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Call ddb.query() with TableName=TABLE_NAME, IndexName=GSI_NAME,
# KeyConditionExpression="tenant_id = :tid",
# ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
# ReturnConsumedCapacity="TOTAL". Store the response in q_resp.

q_items = q_resp.get("Items", [])
q_capacity = q_resp.get("ConsumedCapacity", {}).get("CapacityUnits", 0)
print(f"GSI Query returned {len(q_items)} items, ConsumedCapacity.CapacityUnits = {q_capacity}")

# YOUR CODE: Call ddb.scan() with TableName=TABLE_NAME,
# FilterExpression="tenant_id = :tid",
# ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
# ReturnConsumedCapacity="TOTAL". Store the response in s_resp.

s_items = s_resp.get("Items", [])
s_capacity = s_resp.get("ConsumedCapacity", {}).get("CapacityUnits", 0)
print(f"Scan + Filter returned {len(s_items)} items, ConsumedCapacity.CapacityUnits = {s_capacity}")

print()
print("Both calls return the same logical result (5 items for tenant T1).")
print(f"Query capacity: {q_capacity}, Scan capacity: {s_capacity}")
if q_capacity and s_capacity:
    print(f"Scan uses about {float(s_capacity) / float(q_capacity):.1f}x the capacity of the Query.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: GSI Query ConsumedCapacity.CapacityUnits is strictly less
# than the Scan + FilterExpression ConsumedCapacity.CapacityUnits, for the
# same logical lookup returning the same 5 items.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        ddb_client = boto3.client(
            "dynamodb",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        q_resp = ddb_client.query(
            TableName=TABLE_NAME,
            IndexName=GSI_NAME,
            KeyConditionExpression="tenant_id = :tid",
            ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
            ReturnConsumedCapacity="TOTAL",
        )
        q_count = q_resp.get("Count", 0)
        q_cap = q_resp.get("ConsumedCapacity", {}).get("CapacityUnits")

        s_resp = ddb_client.scan(
            TableName=TABLE_NAME,
            FilterExpression="tenant_id = :tid",
            ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
            ReturnConsumedCapacity="TOTAL",
        )
        s_count = s_resp.get("Count", 0)
        s_cap = s_resp.get("ConsumedCapacity", {}).get("CapacityUnits")

        if q_cap is None or s_cap is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ConsumedCapacity is None on one of the responses. Pass "
                    f"ReturnConsumedCapacity='TOTAL' on both calls."
                ),
            )
        if q_count != 5 or s_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Query returned {q_count} items and Scan returned {s_count} items; "
                    f"both should return 5. Verify the Task 3 load."
                ),
            )
        if float(q_cap) >= float(s_cap):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected Query capacity < Scan capacity. Got Query={q_cap}, Scan={s_cap}. "
                    f"On a 100-item table the gap is unambiguous; if you see this it usually "
                    f"means the Query is hitting the base table instead of the GSI (missing IndexName)."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Three things to verify if the assertion fails: both calls pass `ReturnConsumedCapacity="TOTAL"`, the Query targets the GSI via `IndexName=GSI_NAME`, and the Scan reads the base table. Missing IndexName on the Query is the most common cause of a counter-intuitive capacity ratio.

</details>

<details><summary>Hint 2 (direction)</summary>

Query against the GSI: `KeyConditionExpression` (not `FilterExpression`) is the boundary that limits which items are read. Scan against the base table: `FilterExpression` is applied AFTER every item is read, so the consumed capacity is proportional to the entire table size, not the items returned. The lab's 100-item table makes this easy to see; the same pattern at 1 TB is the difference between a $0.05 query and a $50 query.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE blocks in Task 4:

```python
q_resp = ddb.query(
    TableName=TABLE_NAME,
    IndexName=GSI_NAME,
    KeyConditionExpression="tenant_id = :tid",
    ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
    ReturnConsumedCapacity="TOTAL",
)

s_resp = ddb.scan(
    TableName=TABLE_NAME,
    FilterExpression="tenant_id = :tid",
    ExpressionAttributeValues={":tid": {"S": TARGET_TENANT}},
    ReturnConsumedCapacity="TOTAL",
)
```

</details>

**Wallet check.** A Query returning 5 items consumes ~0.5 RCU at on-demand pricing = about $0.0000001. A Scan reading 100 small items consumes ~12 RCU = about $0.000003. Both well inside the always-cheap range for on-demand. The damage at production scale is what the lab is showing you: same query against a 1 TB table is the difference between pennies and dollars.

## Cleanup

Tear it all down. The cell below deletes the DynamoDB table; the GSI and all 100 items are destroyed with it.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down the DynamoDB table; the GSI is destroyed with it.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** 100 BatchWriteItem operations + 2 Queries + 2 Scans against 100 items lands well under one cent in total. DynamoDB on-demand bills only for what you actually call; the table at rest with no traffic does not accrue per-hour charges.

## Reflection

These are not graded. They are for you.

1. Walk through why a `Scan` with `FilterExpression: tenant_id = :tid` consumes the same capacity as an unfiltered Scan, even though it returns far fewer items. Where does DynamoDB apply the filter in the request lifecycle, and what does that imply about the cost of scanning a 1 TB table for one tenant?

2. Your team is choosing between adding a GSI on a high-cardinality attribute (`user_id`, ~1M unique values) and adding a Local Secondary Index on a low-cardinality attribute (`status`, ~5 unique values). Walk through the trade-offs in terms of partition heat, storage cost, and query flexibility. When is each the right call?

## Exam-Style Practice

**Question 1 (Domain 3, query vs scan capacity model):**

A team is choosing between a DynamoDB `Query` against a GSI and a `Scan` with a `FilterExpression` to retrieve all items for a single tenant from a 10 GB table. The GSI partitions on `tenant_id`. Both calls return the same 50 items. Which statement correctly describes the ConsumedCapacity?

A. The Scan with FilterExpression consumes less capacity because the filter excludes most items before they are read from storage.

B. The Query consumes proportionally to the items returned plus the index key range scanned; the Scan consumes proportionally to the entire table size, because filters are applied after the read.

C. Both calls consume the same capacity because they return the same number of items.

D. The Scan is cheaper if `Select=COUNT` is set, regardless of FilterExpression.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: DynamoDB FilterExpression is applied AFTER items are read from storage, not before. The filter narrows the response but does not reduce the underlying read cost.
- B is correct: Query reads only the items along the key range it touches, charging RCU proportional to items returned plus the scanned key range. Scan reads every item in the table or partition before applying any filter, charging RCU proportional to the entire data scanned. On a 10 GB table the gap is orders of magnitude.
- C is wrong: ConsumedCapacity is driven by data read, not items returned. Query and Scan have fundamentally different read patterns.
- D is wrong: `Select=COUNT` returns only the count, but the Scan still reads every item to determine that count. COUNT does not reduce the per-RCU cost.

</details>

**Question 2 (Domain 3, GSI projection-type trade-off):**

You are adding a GSI on `tenant_id` to a 500 GB DynamoDB table. Every query against the GSI needs three attributes: `tenant_id`, `created_at`, and `display_name`. The table has 40 other attributes that are never accessed via the GSI. Which `ProjectionType` minimizes cost while satisfying the access pattern?

A. `ALL`, because it eliminates follow-up GetItem calls and simplifies the query code.

B. `KEYS_ONLY`, because it minimizes index storage and the missing attributes can be fetched with a GetItem per item.

C. `INCLUDE` with `display_name` projected, because the GSI then returns all three required attributes (tenant_id and created_at are key attributes, automatically projected) without storing the other 40.

D. There is no way to control projection on a GSI; all attributes are projected.

<details><summary>Show answer</summary>

**Correct: C.**

- A works but doubles the index storage for the 40 attributes you never read. On a 500 GB table that translates to a substantial monthly storage bill for no functional benefit.
- B minimizes storage but forces a follow-up GetItem against the base table for every query result, which doubles the read RCU and adds latency. Acceptable for sparse low-cardinality lookups; wrong choice when you know up front which attributes you need.
- C is correct: `INCLUDE` lets you project a named subset of non-key attributes into the GSI. Key attributes (the partition and sort key of the GSI) are always projected automatically. This is the AWS-recommended choice when you know exactly which attributes the access pattern needs.
- D is wrong: DynamoDB supports `ALL`, `KEYS_ONLY`, and `INCLUDE` projection types on every GSI.

</details>